# Module 4: Bring Your Own Data | CSV Connector

This module builds a semantic layer graph from **CSV files describing your own schema** — no BigQuery or cloud data warehouse required.

The CSV connector runs the same ETL pipeline as Module 2:
1. **Extract** — read CSV files describing your tables, columns, and relationships
2. **Transform** — convert to typed graph model objects
3. **Load** — write nodes and relationships into Neo4j
4. **Embed** — generate vector embeddings for semantic search

**What you need:** CSV files in `workshop/datasets/csv/` describing your schema. See Cell 3 for the exact file format.

**Prerequisites:** Complete `workshop/setup/environment-setup.md`. GCP/BigQuery access is **not** required for this module.

## Imports and Environment

**Sync project dependencies before running this notebook!**
```bash
uv sync
```

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

required_vars = [
    'NEO4J_URI', 'NEO4J_USERNAME', 'NEO4J_PASSWORD', 'NEO4J_DATABASE',
    'OPENAI_API_KEY',
]

print("Environment variable check:")
for var in required_vars:
    val = os.getenv(var)
    status = '✅' if val else '❌'
    print(f"  {status} {var}")

Environment variable check:
  ✅ NEO4J_URI
  ✅ NEO4J_USERNAME
  ✅ NEO4J_PASSWORD
  ✅ NEO4J_DATABASE
  ✅ OPENAI_API_KEY


## Confirm Connections

In [2]:
from neo4j import GraphDatabase
from openai import OpenAI

neo4j_driver = GraphDatabase.driver(
    uri=os.getenv('NEO4J_URI'),
    auth=(os.getenv('NEO4J_USERNAME'), os.getenv('NEO4J_PASSWORD')),
)
neo4j_database = os.getenv('NEO4J_DATABASE', 'neo4j')

# Verify Neo4j connection
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run('RETURN 1 AS ping')
    message = '✅' if result.single()['ping'] else '❌'
    print(f"{message} Neo4j driver connected")

# Initialize OpenAI client
embedding_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print("✅ OpenAI client initialized")

✅ Neo4j driver connected
✅ OpenAI client initialized


## CSV File Format

The CSV connector reads a directory of CSV files, each describing one component of the semantic layer. 

The `neocarta` library will generate unique IDs for each entity based on the hierarchical name fields. 

---

### Core schema files (required to run the semantic layer)

**`database_info.csv`** — one row per database
| Column | Required | Notes |
|--------|----------|-------|
| `database_name` | ✅ | Identifier used to link all other files |
| `description` | | Used for embeddings — describe what this database contains |

**`schema_info.csv`** — one row per schema (or one row if your DB has no schemas)
| Column | Required | Notes |
|--------|----------|-------|
| `database_name` | ✅ | Must match `database_info.csv` |
| `schema_name` | ✅ | |
| `description` | | |

**`table_info.csv`** — one row per table
| Column | Required | Notes |
|--------|----------|-------|
| `database_name` | ✅ | |
| `schema_name` | ✅ | |
| `table_name` | ✅ | |
| `description` | | Used for embeddings — describe what this table tracks |

**`column_info.csv`** — one row per column
| Column | Required | Notes |
|--------|----------|-------|
| `database_name` | ✅ | |
| `schema_name` | ✅ | |
| `table_name` | ✅ | |
| `column_name` | ✅ | |
| `description` | | Used for embeddings |
| `data_type` | | e.g. `INTEGER`, `VARCHAR`, `TIMESTAMP` |
| `is_primary_key` | | `True` / `False` |
| `is_foreign_key` | | `True` / `False` |
| `is_nullable` | | `True` / `False` |

**`column_references_info.csv`** — one row per foreign key relationship
| Column | Required |
|--------|----------|
| `source_database_name` | ✅ |
| `source_schema_name` | ✅ |
| `source_table_name` | ✅ |
| `source_column_name` | ✅ |
| `target_database_name` | ✅ |
| `target_schema_name` | ✅ |
| `target_table_name` | ✅ |
| `target_column_name` | ✅ |

**`value_info.csv`** — one row per enumerated column value
| Column | Required | Notes |
|--------|----------|-------|
| `database_name` | ✅ | |
| `schema_name` | ✅ | |
| `table_name` | ✅ | |
| `column_name` | ✅ | |
| `value` | ✅ | The discrete value (e.g. `active`, `cancelled`, `pending`) |

Useful for categorical columns where knowing the valid values helps the agent filter correctly. Creates `Value` nodes linked to their parent `Column` via `HAS_VALUE`.

---

### Enrichment files (optional — adds business terminology)

**`glossary_info.csv`** — one row per glossary
| Column | Required |
|--------|----------|
| `glossary_name` | ✅ |
| `description` | |

**`category_info.csv`** — groups terms within a glossary
| Column | Required |
|--------|----------|
| `glossary_name` | ✅ |
| `category_name` | ✅ |
| `description` | |

**`business_term_info.csv`** — individual business terms
| Column | Required |
|--------|----------|
| `glossary_name` | ✅ |
| `category_name` | ✅ |
| `term_name` | ✅ |
| `description` | |

**`table_term_info.csv`** — links tables to business terms (creates `TAGGED_WITH` edges)
| Column | Required |
|--------|----------|
| `database_name` | ✅ |
| `schema_name` | ✅ |
| `table_name` | ✅ |
| `glossary_name` | ✅ |
| `category_name` | ✅ |
| `term_name` | ✅ |

**`column_term_info.csv`** — links columns to business terms (creates `TAGGED_WITH` edges)
| Column | Required |
|--------|----------|
| `database_name` | ✅ |
| `schema_name` | ✅ |
| `table_name` | ✅ |
| `column_name` | ✅ |
| `glossary_name` | ✅ |
| `category_name` | ✅ |
| `term_name` | ✅ |

## Inspect Your CSV Files

Check which files are present in the CSV directory and preview their contents.

In [8]:
import pandas as pd
from pathlib import Path

# Define the CSV Directory path. 
csv_dir = Path('../datasets/user/csv/')

csv_files = sorted(csv_dir.glob('*.csv'))
if not csv_files:
    print(f"⚠️  No CSV files found in {csv_dir}")
    print("Add your schema CSV files to workshop/datasets/csv/ and re-run this cell.")
else:
    print(f"Found {len(csv_files)} CSV file(s) in {csv_dir}:\n")
    for path in csv_files:
        df = pd.read_csv(path)
        print(f"{path.name}  ({len(df)} rows)")

⚠️  No CSV files found in ../datasets/user/csv
Add your schema CSV files to workshop/datasets/csv/ and re-run this cell.


## Run the CSV Connector

The `CSVConnector` runs the same extract → transform → load pipeline as Module 2, reading from your CSV files instead of BigQuery.

Files that don't exist in the directory are skipped automatically. You don't need all files to run a useful semantic layer.

In [5]:
from neocarta.connectors.csv import CSVConnector

In [6]:
csv_connector = CSVConnector(
    csv_directory=str(csv_dir),
    neo4j_driver=neo4j_driver,
    database_name=neo4j_database,
)

In [7]:
csv_connector.run()
print("CSV ETL workflow complete.")

Extracting metadata from CSV files...
Extracting CSV files from /var/folders/7_/vqs74z3j5hscgzbt0ydbmqyw0000gq/T/tmpsrsnu7sr...
  Extracted 1 rows from database_info.csv
  Extracted 1 rows from schema_info.csv
  Extracted 33 rows from table_info.csv
  Extracted 327 rows from column_info.csv
  Extracted 54 rows from column_references_info.csv
  Skipping value_info.csv (file not found)
  Skipping query_info.csv (file not found)
  Skipping query_table_info.csv (file not found)
  Skipping query_column_info.csv (file not found)
  Extracted 1 rows from glossary_info.csv
  Extracted 9 rows from category_info.csv
  Extracted 76 rows from business_term_info.csv
  Extracted 86 rows from column_term_info.csv
  Extracted 32 rows from table_term_info.csv
Transforming metadata...
Loading metadata into Neo4j...

=== Loading Nodes ===
Loading 1 database nodes...
{'_contains_updates': True, 'labels_added': 1, 'nodes_created': 1, 'properties_set': 3}
Loading 1 schema nodes...
{'_contains_updates': True,

## Generate Vector Embeddings

Embeddings are generated from the `description` fields on your nodes. The quality of your descriptions directly determines semantic search accuracy.

In [9]:
from neocarta.enrichment.embeddings import OpenAIEmbeddingsConnector

node_labels = ['Table', 'Column', 'BusinessTerm']

openai_embedding_connector = OpenAIEmbeddingsConnector(
    client=embedding_client,
    embedding_model='text-embedding-3-small',
    dimensions=768,
    neo4j_driver=neo4j_driver,
    database_name=neo4j_database,
)

print(f"Generating embeddings for: {node_labels}")
openai_embedding_connector.run(node_labels=node_labels)
print("\nEmbeddings complete!")

Generating embeddings for: ['Table', 'Column', 'BusinessTerm']
Processing Table nodes...
--------------------------------
Processing batch 1 of 1  
Successful Embeddings : 33
{}
Processing Column nodes...
--------------------------------
Processing batch 1 of 4  
Processing batch 2 of 4  
Processing batch 3 of 4  
Processing batch 4 of 4  
Successful Embeddings : 327
{}
Processing BusinessTerm nodes...
--------------------------------
Processing batch 1 of 1  
Successful Embeddings : 81
{}

Embeddings complete!


In [11]:
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run("""
        MATCH (n)
        WHERE n.embedding IS NOT NULL OR n.embedding IS NULL
        WITH labels(n)[0] AS label, n
        RETURN label,
               count(n) AS total,
               count(n.embedding) AS with_embedding
        ORDER BY label
    """)
    print("Embedding coverage:")
    for record in result:
        status = '✅' if record['total'] == record['with_embedding'] else '⚠️'
        print(f"  {status} {record['label']}: {record['with_embedding']}/{record['total']} nodes embedded")

Embedding coverage:
  ✅ BusinessTerm: 81/81 nodes embedded
  ⚠️ Category: 0/9 nodes embedded
  ✅ Column: 327/327 nodes embedded
  ⚠️ Database: 0/1 nodes embedded
  ⚠️ Glossary: 0/1 nodes embedded
  ⚠️ Schema: 0/1 nodes embedded
  ✅ Table: 33/33 nodes embedded


## Explore the Graph

Verify the schema structure that was loaded into Neo4j.

In [17]:
# Show all tables and columns
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run("""
        MATCH (t:Table)-[:HAS_COLUMN]->(c:Column)
        RETURN t.name AS table,
               collect({name: c.name, type: c.type, pk: c.is_primary_key, fk: c.is_foreign_key}) AS columns
        ORDER BY t.name
    """)
    print("Sample schema loaded into Neo4j:")
    print("-" * 60)
    for record in list(result)[:2]:
        print(f"\nTable: {record['table']}")
        for col in record['columns'][:5]:
            flags = []
            if col.get('pk'): flags.append('PK')
            if col.get('fk'): flags.append('FK')
            flag_str = f" [{', '.join(flags)}]" if flags else ""
            col_type = f" ({col['type']})" if col.get('type') else ""
            print(f"  - {col['name']}{col_type}{flag_str}")

Sample schema loaded into Neo4j:
------------------------------------------------------------

Table: campaigns
  - budget_usd (NUMERIC)
  - campaign_id (STRING) [PK]
  - channel (STRING)
  - end_date (DATE)
  - name (STRING)

Table: compensation
  - approved_by (STRING)
  - base_salary (NUMERIC)
  - bonus_target_pct (NUMERIC)
  - change_type (STRING)
  - compensation_id (STRING) [PK]


In [20]:
# Show foreign key relationships
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run("""
        MATCH (c1:Column)-[:REFERENCES]->(c2:Column)
        MATCH (c1)<-[:HAS_COLUMN]-(t1:Table)
        MATCH (c2)<-[:HAS_COLUMN]-(t2:Table)
        RETURN t1.name AS from_table, c1.name AS from_col,
               t2.name AS to_table, c2.name AS to_col
        ORDER BY from_table
    """)
    rows = result.data()
    if rows:
        print("Sample foreign key relationships:")
        for r in rows[:5]:
            print(f"  {r['from_table']}.{r['from_col']} → {r['to_table']}.{r['to_col']}")
    else:
        print("No foreign key relationships found.")
        print("Add a column_references_info.csv to define join paths between tables.")

Sample foreign key relationships:
  campaigns.owner_employee_id → employees.employee_id
  compensation.employee_id → employees.employee_id
  customer_addresses.customer_id → customers.customer_id
  customer_contacts.customer_id → customers.customer_id
  customers.account_owner_id → employees.employee_id


In [ ]:
# Show vector indexes
with neo4j_driver.session(database=neo4j_database) as session:
    result = session.run("SHOW VECTOR INDEXES")
    rows = result.data()
    if rows:
        print("Vector indexes:")
        for r in rows:
            print(f"  - {r['name']}: label={r.get('labelsOrTypes', 'N/A')[0]}, state={r['state']}")
    else:
        print("No vector indexes found. Check that embeddings ran successfully.")

Vector indexes:
  - businessterm_vector_index: label=BusinessTerm, state=ONLINE
  - column_vector_index: label=Column, state=ONLINE
  - table_vector_index: label=Table, state=ONLINE


## Tips for Better Results

**Column descriptions are the most important input.** The semantic search works by comparing your question's embedding against column description embeddings. A column with no description will never match.

Good description: `"Total revenue generated by the order, calculated as the sum of all line item prices"`  
Weak description: `"Revenue"`

**Foreign keys determine join quality.** If you omit `column_references_info.csv`, the agent will still find the right columns but may struggle with multi-table joins. Explicit FK paths remove ambiguity.

**Iterating:** After editing your CSV files, re-run Cell 5 (the connector) and Cell 6 (embeddings). The connector uses `MERGE` — it's safe to re-run without creating duplicates.